# Model 1: DeBERTa-v3-large extractive QA

**The question:** does a bigger and stronger reader beat our Milestone 2 strong baseline (RoBERTa-base, 125M parameters, test BLEU 0.384), and on which spoiler types?

This notebook is the Milestone 2 QA pipeline with one change: the reader is **`deepset/deberta-v3-large-squad2`** (DeBERTa-v3-large, 435M parameters, already fine-tuned on SQuAD 2.0). DeBERTa-v3-large is also what the SemEval-2023 organizers used for their baseline (0.40 test BLEU). Everything else stays the same, so the comparison isolates the reader:
- the same 384-token windows with 128-token overlap, one training example per spoiler span;
- the same decoding (best span for phrase/passage, top-k non-overlapping spans for multi), with the per-type length caps and k tuned on dev;
- the **same spoiler types**: the Milestone 2 RoBERTa type classifier's predictions, copied into `inputs/milestone2/`.

**Learning-rate sweep:** 5e-6, 1e-5, 2e-5 (large models want smaller rates than base ones). Batch 8 with 2 accumulation steps (effective 16), 3 epochs, AdamW with weight decay 0.01, linear schedule with 10% warmup, bf16. The best epoch is chosen by dev BLEU decoded with gold types, as in Milestone 2.

**Running it on the L4 VM** (about 3.5 hours: each learning rate takes about 66 minutes):
```bash
cd ~/project/milestone3
source ~/project/.venv/bin/activate
pip install -r requirements.txt               # once for all Milestone 3 notebooks
cp -r ../milestone2/tools .                   # the official METEOR jar, already downloaded in Milestone 2
tmux new -s m3                                # detach with Ctrl-b d, reattach with: tmux attach -t m3
jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=-1 1-deberta-qa.ipynb
```
`bash run-all.sh` runs all Milestone 3 notebooks one after the other instead. A learning rate that has finished (its `runs/.../run_info.json` exists) is skipped when the notebook is run again, so an interrupted sweep picks up where it stopped.

**Smoke test** (a few minutes on a laptop CPU; on the VM it uses the GPU): `SMOKE=1 jupyter nbconvert --to notebook --execute 1-deberta-qa.ipynb --output smoke-run.ipynb`. It swaps in a tiny model, trains 2 steps on 16 posts and writes everything under `smoke/`.

**Outputs:** checkpoints in `runs/deberta-qa-lr*/` (not committed); `output/deberta-qa-sweep.json`, `output/deberta-qa-settings.json`, predictions `output/deberta-qa-{dev,test}.jsonl` and `output/deberta-qa-goldtypes-test.jsonl`, and scores in `output/scores/`.

## Setup

Imports and run settings. The shared helpers live in the `spoilerqa/` package next to this notebook (a copy of the Milestone 2 package plus `gen.py` for the generative models). `SMOKE=1` in the environment switches to the tiny test run, which uses the tiny DeBERTa-v3-xsmall instead of the large model.

In [1]:
import gc
import json
import os
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForQuestionAnswering, AutoTokenizer, Trainer, TrainingArguments, set_seed
from transformers.utils import logging as hf_logging

from spoilerqa.data import DEFAULT_DATA_DIR, TYPES, load_jsonl, load_split, post_text, spoiler_type, write_jsonl
from spoilerqa.metrics import as_text, bleu, ensure_nltk_data

hf_logging.set_verbosity_error()
ensure_nltk_data()
pd.set_option("display.max_colwidth", 100)

SMOKE = os.environ.get("SMOKE", "0") == "1"
DATA_DIR = Path(os.environ.get("DATA_DIR", DEFAULT_DATA_DIR))
WORK_DIR = Path("smoke") if SMOKE else Path(".")
RUNS_DIR, OUTPUT_DIR = WORK_DIR / "runs", WORK_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
M2_DIR = Path("inputs/milestone2")  # frozen copies of the Milestone 2 outputs we reuse
SEED = 42

HAS_GPU = torch.cuda.is_available()
USE_BF16 = HAS_GPU and torch.cuda.is_bf16_supported()
DEVICE = "cuda" if HAS_GPU else "cpu"  # a laptop runs the smoke test on the CPU

# Smoke-test limits (None / -1 = use everything).
MAX_TRAIN = 16 if SMOKE else None
MAX_EVAL = 8 if SMOKE else None
MAX_STEPS = 2 if SMOKE else -1

from spoilerqa.qa import (DEFAULT_MAX_ANSWER_TOKENS, FeatureDataset, candidate_spans, decode, inference_features,
                          training_examples, training_features)

NAME = "deberta-qa"
MODEL = "microsoft/deberta-v3-xsmall" if SMOKE else "deepset/deberta-v3-large-squad2"
LEARNING_RATES = [5e-6, 1e-5, 2e-5]
EPOCHS, BATCH_SIZE, GRAD_ACCUM = 3, (2 if SMOKE else 8), (1 if SMOKE else 2)
MAX_LENGTH, STRIDE = 384, 128

print("GPU:", torch.cuda.get_device_name(0) if HAS_GPU else "none", "| bf16:", USE_BF16, "| smoke test:", SMOKE,
      "| model:", MODEL)

/home/dhanush/project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU: NVIDIA L4 | bf16: True | smoke test: False | model: deepset/deberta-v3-large-squad2


/home/dhanush/project/.venv/lib/python3.12/site-packages/nltk/downloader.py:1076: UserWarning: NLTK will not authorize the non-private download directory '/home/dhanush/nltk_data': it (or an ancestor) is world- or group-writable, so another local user could plant files there. Choose a private location such as ~/nltk_data.
  for msg in self.incr_download(info_or_id, download_dir, force):


Load the splits, and the spoiler types predicted by the Milestone 2 classifier for dev and test (test balanced accuracy 0.692). Using the same types as the strong baseline means any difference in BLEU comes from the reader.

In [2]:
train = load_split(DATA_DIR, "train")[:MAX_TRAIN]
dev = load_split(DATA_DIR, "dev")[:MAX_EVAL]
test = load_split(DATA_DIR, "test")[:MAX_EVAL]
print({"train": len(train), "dev": len(dev), "test": len(test)})

m2_types = {split: {p["uuid"]: p["spoilerType"] for p in load_jsonl(M2_DIR / f"type-classifier-{split}.jsonl")}
            for split in ["dev", "test"]}
predicted_types = {"dev": [m2_types["dev"][r["uuid"]] for r in dev],
                   "test": [m2_types["test"][r["uuid"]] for r in test]}

{'train': 3157, 'dev': 794, 'test': 1000}


Turn the posts into training windows, exactly as in Milestone 2:
- the context is the title followed by the paragraphs, one per line, split into windows of 384 tokens that overlap by 128;
- each training window is labeled with the answer's start and end tokens, or with the first token if it doesn't contain the answer;
- a multi spoiler becomes one training example per span, and the few spans we can't find in the article are skipped.

DeBERTa's tokenizer counts the space before a word as part of the token, so a decoded span can start with a space. The decoder strips it.

In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)
examples, skipped = training_examples(train)
train_enc = training_features(tokenizer, examples, MAX_LENGTH, STRIDE)
qa_keys = [k for k in ["input_ids", "attention_mask", "token_type_ids"] if k in train_enc]
train_ds = FeatureDataset(train_enc, qa_keys + ["start_positions", "end_positions"])
dev_enc, dev_contexts = inference_features(tokenizer, dev, MAX_LENGTH, STRIDE)
dev_ds = FeatureDataset(dev_enc, qa_keys)

answered = sum(s > 0 for s in train_enc["start_positions"])
print(f"train: {len(train)} posts -> {len(examples)} spans ({skipped} not found in the article) -> "
      f"{len(train_ds)} windows ({answered} contain the answer)")
print(f"dev: {len(dev)} posts -> {len(dev_ds)} windows")

train: 3157 posts -> 4533 spans (14 not found in the article) -> 15043 windows (5316 contain the answer)
dev: 794 posts -> 2282 windows


The trainer from Milestone 2: after every epoch it decodes the dev spoilers and reports **BLEU** instead of a loss. It decodes with the *gold* types, so this score measures extraction only; the type classifier comes in at prediction time.

In [4]:
class SpoilerQATrainer(Trainer):
    # Trainer whose evaluation decodes dev spoilers and reports BLEU instead of a loss.
    def __init__(self, *args, dev_records=None, dev_enc=None, dev_contexts=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.dev_records, self.dev_enc, self.dev_contexts = dev_records, dev_enc, dev_contexts

    def evaluate(self, eval_dataset=None, ignore_keys=None, metric_key_prefix="eval"):
        loader = self.get_eval_dataloader(eval_dataset or self.eval_dataset)
        output = self.evaluation_loop(loader, description="Decoding dev", prediction_loss_only=False,
                                      metric_key_prefix=metric_key_prefix)
        start_logits, end_logits = output.predictions
        candidates = candidate_spans(self.dev_enc, self.dev_contexts, start_logits, end_logits)
        preds = [as_text(decode(c, ctx, spoiler_type(r)))
                 for c, ctx, r in zip(candidates, self.dev_contexts, self.dev_records)]
        truths = [as_text(r["spoiler"]) for r in self.dev_records]
        metrics = {f"{metric_key_prefix}_bleu": bleu(truths, preds)}
        self.log(metrics)
        self.control = self.callback_handler.on_evaluate(self.args, self.state, self.control, metrics)
        return metrics

One training run for one learning rate. It keeps the epoch with the best dev BLEU, saves that model to `runs/deberta-qa-lr<lr>/best` and writes the per-epoch scores and the training time to `run_info.json` in the same folder. If that file already exists, the run is skipped and its saved scores are reused.

In [5]:
def free_gpu():
    gc.collect()
    if HAS_GPU:
        torch.cuda.empty_cache()



def epoch_or_steps():
    # Evaluate and save after every epoch; the smoke test does it after its 2 steps instead.
    if MAX_STEPS < 0:
        return dict(eval_strategy="epoch", save_strategy="epoch")
    return dict(eval_strategy="steps", eval_steps=MAX_STEPS, save_strategy="steps", save_steps=MAX_STEPS)


def train_qa(learning_rate):
    run_dir = RUNS_DIR / f"{NAME}-lr{learning_rate:g}"
    if (run_dir / "run_info.json").exists():
        print("already trained, reusing", run_dir / "run_info.json")
        return json.loads((run_dir / "run_info.json").read_text())
    set_seed(SEED)
    model = AutoModelForQuestionAnswering.from_pretrained(MODEL)
    args = TrainingArguments(
        output_dir=str(run_dir), learning_rate=learning_rate, num_train_epochs=EPOCHS, max_steps=MAX_STEPS,
        per_device_train_batch_size=BATCH_SIZE, gradient_accumulation_steps=GRAD_ACCUM, per_device_eval_batch_size=32,
        weight_decay=0.01, warmup_steps=0.1, lr_scheduler_type="linear",  # warmup_steps < 1 means a ratio
        **epoch_or_steps(), save_total_limit=2, save_only_model=True,
        load_best_model_at_end=True, metric_for_best_model="bleu", greater_is_better=True,
        logging_steps=50, bf16=USE_BF16, use_cpu=not HAS_GPU, report_to="none", seed=SEED,
    )
    trainer = SpoilerQATrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=dev_ds,
                               processing_class=tokenizer, dev_records=dev, dev_enc=dev_enc, dev_contexts=dev_contexts)
    trainer.train()
    trainer.save_model(str(run_dir / "best"))
    history = trainer.state.log_history
    per_epoch = [h["eval_bleu"] for h in history if "eval_bleu" in h]
    info = {"learning_rate": learning_rate, "dev_bleu": max(per_epoch), "dev_bleu_per_epoch": per_epoch,
            "train_minutes": round(next(h["train_runtime"] for h in history if "train_runtime" in h) / 60, 1),
            "run_dir": str(run_dir)}
    (run_dir / "run_info.json").write_text(json.dumps(info, indent=2) + "\n")
    del trainer, model
    free_gpu()
    return info

**The learning-rate sweep.** The table shows dev BLEU (decoded with gold types) after every epoch, and is saved to `output/deberta-qa-sweep.json` for the comparison notebook and the report.

In [6]:
sweep = []
for lr in LEARNING_RATES:
    print(f"=== learning rate {lr:g} ===")
    sweep.append(train_qa(lr))

(OUTPUT_DIR / f"{NAME}-sweep.json").write_text(json.dumps(sweep, indent=2) + "\n")
pd.DataFrame(sweep)[["learning_rate", "dev_bleu", "dev_bleu_per_epoch", "train_minutes"]]

=== learning rate 5e-06 ===


/home/dhanush/project/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
Loading weights: 100%|██████████| 392/392 [00:00<00:00, 3279.90it/s]


{'loss': '2.43', 'grad_norm': '30.85', 'learning_rate': '8.657e-07', 'epoch': '0.05316'}
{'loss': '1.929', 'grad_norm': '28.63', 'learning_rate': '1.749e-06', 'epoch': '0.1063'}
{'loss': '1.572', 'grad_norm': '28.01', 'learning_rate': '2.633e-06', 'epoch': '0.1595'}
{'loss': '1.453', 'grad_norm': '17.52', 'learning_rate': '3.516e-06', 'epoch': '0.2127'}
{'loss': '1.4', 'grad_norm': '18.2', 'learning_rate': '4.399e-06', 'epoch': '0.2658'}
{'loss': '1.274', 'grad_norm': '14.86', 'learning_rate': '4.969e-06', 'epoch': '0.319'}
{'loss': '1.342', 'grad_norm': '18.03', 'learning_rate': '4.87e-06', 'epoch': '0.3721'}
{'loss': '1.27', 'grad_norm': '18.33', 'learning_rate': '4.772e-06', 'epoch': '0.4253'}
{'loss': '1.197', 'grad_norm': '13.95', 'learning_rate': '4.673e-06', 'epoch': '0.4785'}
{'loss': '1.217', 'grad_norm': '13.83', 'learning_rate': '4.575e-06', 'epoch': '0.5316'}
{'loss': '1.184', 'grad_norm': '19.52', 'learning_rate': '4.476e-06', 'epoch': '0.5848'}
{'loss': '1.201', 'grad_nor

Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.07s/it]


{'loss': '1.135', 'grad_norm': '14', 'learning_rate': '3.689e-06', 'epoch': '1.01'}
{'loss': '0.9002', 'grad_norm': '13.83', 'learning_rate': '3.591e-06', 'epoch': '1.063'}
{'loss': '1.075', 'grad_norm': '14.92', 'learning_rate': '3.492e-06', 'epoch': '1.116'}
{'loss': '0.9259', 'grad_norm': '16.07', 'learning_rate': '3.394e-06', 'epoch': '1.169'}
{'loss': '0.9963', 'grad_norm': '10.7', 'learning_rate': '3.295e-06', 'epoch': '1.222'}
{'loss': '1.073', 'grad_norm': '10.9', 'learning_rate': '3.197e-06', 'epoch': '1.275'}
{'loss': '0.9633', 'grad_norm': '12.29', 'learning_rate': '3.098e-06', 'epoch': '1.329'}
{'loss': '1.009', 'grad_norm': '12.12', 'learning_rate': '3e-06', 'epoch': '1.382'}
{'loss': '1.023', 'grad_norm': '14.49', 'learning_rate': '2.902e-06', 'epoch': '1.435'}
{'loss': '0.9593', 'grad_norm': '19.74', 'learning_rate': '2.803e-06', 'epoch': '1.488'}
{'loss': '0.9976', 'grad_norm': '11.36', 'learning_rate': '2.705e-06', 'epoch': '1.541'}
{'loss': '0.995', 'grad_norm': '18.4

Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.09s/it]


... (21 outputs left out)

Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.13s/it]


{'train_runtime': '3958', 'train_samples_per_second': '11.4', 'train_steps_per_second': '0.713', 'train_loss': '0.876', 'epoch': '3'}


Writing model shards: 100%|██████████| 1/1 [00:06<00:00,  6.93s/it]


,learning_rate,dev_bleu,dev_bleu_per_epoch,train_minutes
0,0.000005,0.485686,"[0.46843, 0.480859, 0.485686]",66.0
1,0.000010,0.496414,"[0.473275, 0.483333, 0.496414]",66.0
2,0.000020,0.488672,"[0.476192, 0.488031, 0.488672]",66.0


## Prediction

Load the model with the best dev BLEU, score every dev and test window once, and keep all plausible answer spans per post (up to 128 tokens), ranked by start + end score. Decoding then picks from these candidates, so trying decoding settings needs no extra model calls.

In [7]:
best_run = max(sweep, key=lambda info: info["dev_bleu"])
print("best learning rate:", best_run["learning_rate"], "| dev BLEU (gold types):", round(best_run["dev_bleu"], 4))
model = AutoModelForQuestionAnswering.from_pretrained(Path(best_run["run_dir"]) / "best").to(DEVICE).eval()


def score_windows(enc, batch_size=32):
    starts, ends = [], []
    with torch.no_grad(), torch.autocast(device_type=DEVICE, dtype=torch.bfloat16, enabled=USE_BF16):
        for i in range(0, len(enc["input_ids"]), batch_size):
            batch = {k: torch.tensor(enc[k][i:i + batch_size]).to(DEVICE) for k in qa_keys}
            out = model(**batch)
            starts.append(out.start_logits.float().cpu().numpy())
            ends.append(out.end_logits.float().cpu().numpy())
    return np.concatenate(starts), np.concatenate(ends)


candidates, contexts = {}, {}
for split, records in [("dev", dev), ("test", test)]:
    enc, contexts[split] = inference_features(tokenizer, records, MAX_LENGTH, STRIDE)
    candidates[split] = candidate_spans(enc, contexts[split], *score_windows(enc), max_answer_tokens=128)
print({split: len(c) for split, c in candidates.items()}, "posts with candidates")

best learning rate: 1e-05 | dev BLEU (gold types): 0.4964


Loading weights: 100%|██████████| 392/392 [00:01<00:00, 377.10it/s]


{'dev': 794, 'test': 1000} posts with candidates


**Tuning decoding on dev**, with the same grid as Milestone 2: the longest answer allowed per type, and the number of spans for multi. For each predicted type we keep the setting with the best dev BLEU over the posts predicted as that type, using the classifier's types as at test time. Test is never used here.

In [8]:
CAP_GRID = {"phrase": [8, 16, 24, 32], "passage": [48, 64, 96, 128], "multi": [24, 48, 64]}
K_GRID = [1, 2, 3, 4, 5, 6]

settings = {"max_answer_tokens": dict(DEFAULT_MAX_ANSWER_TOKENS), "k": 5}
tuning = []
for t in TYPES:
    idx = [i for i, p in enumerate(predicted_types["dev"]) if p == t]
    if not idx:
        continue
    truths = [as_text(dev[i]["spoiler"]) for i in idx]
    for cap in CAP_GRID[t]:
        for k in (K_GRID if t == "multi" else [settings["k"]]):
            caps = {**settings["max_answer_tokens"], t: cap}
            preds = [as_text(decode(candidates["dev"][i], contexts["dev"][i], t, caps, k)) for i in idx]
            tuning.append({"type": t, "max_answer_tokens": cap, "k": k if t == "multi" else None,
                           "dev_posts": len(idx), "dev_bleu": bleu(truths, preds)})
    best = max((row for row in tuning if row["type"] == t), key=lambda row: row["dev_bleu"])
    settings["max_answer_tokens"][t] = best["max_answer_tokens"]
    if t == "multi":
        settings["k"] = best["k"]

settings.update({"model": MODEL, "learning_rate": best_run["learning_rate"]})
(OUTPUT_DIR / f"{NAME}-settings.json").write_text(json.dumps(settings, indent=2) + "\n")
print(json.dumps(settings, indent=2))
pd.DataFrame(tuning).sort_values(["type", "dev_bleu"], ascending=[True, False]).groupby("type").head(3)

{
  "max_answer_tokens": {
    "phrase": 16,
    "passage": 64,
    "multi": 24
  },
  "k": 5,
  "model": "deepset/deberta-v3-large-squad2",
  "learning_rate": 1e-05
}


,type,max_answer_tokens,k,dev_posts,dev_bleu
12,multi,24,5.0,191,0.330786
18,multi,48,5.0,191,0.317869
24,multi,64,5.0,191,0.303971
5,passage,64,NaN,316,0.383320
7,passage,128,NaN,316,0.381591
4,passage,48,NaN,316,0.380928
1,phrase,16,NaN,287,0.610894
2,phrase,24,NaN,287,0.609730
3,phrase,32,NaN,287,0.608563


Write the predictions in the SemEval run format (`uuid`, `spoilerType`, `spoiler`):
- `deberta-qa-{dev,test}.jsonl` uses the Milestone 2 classifier's types: **this is the system we compare.**
- `deberta-qa-goldtypes-test.jsonl` decodes with the gold types, for analysis only.

In [9]:
def predictions(records, split, types):
    return [{"uuid": r["uuid"], "spoilerType": t,
             "spoiler": decode(c, ctx, t, settings["max_answer_tokens"], settings["k"])}
            for r, c, ctx, t in zip(records, candidates[split], contexts[split], types)]


write_jsonl(predictions(dev, "dev", predicted_types["dev"]), OUTPUT_DIR / f"{NAME}-dev.jsonl")
write_jsonl(predictions(test, "test", predicted_types["test"]), OUTPUT_DIR / f"{NAME}-test.jsonl")
write_jsonl(predictions(test, "test", [spoiler_type(r) for r in test]), OUTPUT_DIR / f"{NAME}-goldtypes-test.jsonl")
del model
free_gpu()
print(sorted(p.name for p in OUTPUT_DIR.glob(f"{NAME}*")))

['deberta-qa-dev.jsonl', 'deberta-qa-goldtypes-test.jsonl', 'deberta-qa-settings.json', 'deberta-qa-sweep.json', 'deberta-qa-test.jsonl']


## Scores

`score.py` (the Milestone 2 evaluation script, copied here) scores each prediction file exactly as from the command line: BLEU (the main number), METEOR, BERTScore, exact match and token F1, overall and per gold type, plus type balanced accuracy. The JSON files go to `output/scores/`. The smoke test skips BERTScore and scores only the posts it predicted.

In [10]:
def gold_file(split, records):
    if not SMOKE:
        return DATA_DIR / f"{split}.jsonl"
    path = WORK_DIR / f"gold-{split}.jsonl"  # the smoke test predicts only a few posts
    write_jsonl(records, path)
    return path


def run_score(split, records, name):
    cmd = [sys.executable, "score.py", "--gold", str(gold_file(split, records)),
           "--pred", str(OUTPUT_DIR / f"{name}-{split}.jsonl"),
           "--output", str(OUTPUT_DIR / "scores" / f"{name}-{split}.json")]
    if SMOKE:
        cmd.append("--no-bertscore")
    print(f"$ python score.py --gold {cmd[3]} --pred {cmd[5]}")
    print(subprocess.run(cmd, capture_output=True, text=True, check=True).stdout)


run_score("dev", dev, NAME)
run_score("test", test, NAME)
run_score("test", test, f"{NAME}-goldtypes")

$ python score.py --gold /home/dhanush/project/data/splits/dev.jsonl --pred output/deberta-qa-dev.jsonl
Scored 794 posts (0 missing predictions)

spoiler              all    phrase   passage     multi
n                    794       333       318       143
bleu              0.4529    0.5906    0.3341    0.3966
meteor            0.4919    0.3057    0.4807    0.5918
bertscore         0.9238    0.9425    0.9084    0.9146
exact-match       0.3149    0.5375    0.1698    0.1189
token-f1          0.5610    0.6518    0.4672    0.5583

spoiler type: balanced accuracy 0.7365
  phrase   precision 0.7840  recall 0.6757  f1 0.7258
  passage  precision 0.7342  recall 0.7296  f1 0.7319
  multi    precision 0.6021  recall 0.8042  f1 0.6886

BLEU (all): 0.4529

$ python score.py --gold /home/dhanush/project/data/splits/test.jsonl --pred output/deberta-qa-test.jsonl
Scored 1000 posts (0 missing predictions)

spoiler              all    phrase   passage     multi
n                   1000       423       4

## A few test predictions

Six random test posts with their gold and predicted spoilers, to sanity-check the numbers above.

In [11]:
test_preds = {p["uuid"]: p for p in load_jsonl(OUTPUT_DIR / f"{NAME}-test.jsonl")}
sample = pd.DataFrame([{
    "post": post_text(r), "gold type": spoiler_type(r), "predicted type": test_preds[r["uuid"]]["spoilerType"],
    "gold spoiler": as_text(r["spoiler"]), "predicted spoiler": as_text(test_preds[r["uuid"]]["spoiler"]),
} for r in test])
sample.sample(min(6, len(sample)), random_state=0)

,post,gold type,predicted type,gold spoiler,predicted spoiler
993,Inside our three-month effort to attend Apple's iPhone 7 launch party,passage,passage,"The truth though is that large tech companies, especially in Silicon Valley, often use access to...",Apple officially announces that – guess what – it actually will have an event. And on 7 September.
859,Mum takes 'funny' photo of toddler - breaks down over chilling truth behind it,passage,passage,"""She was practising for a lockdown drill at her preschool and what you should do if you are stuc...","her little girl was practising hiding from a gunman, should one enter her nursery"
298,Duterte plans to ‘SHOOT and KILL’ these people; find out who they are,passage,multi,business owners that he will kill them if they continue to practice endo or contractualization,business owners that he will kill them if they continue to practice endo or contractualization P...
553,LOOK: This Is Definitely The Biggest Spring 2014 Trend,passage,phrase,popping Spring colors and mixed prints,popping Spring colors and mixed prints
672,Lot of Pokémon Go players demanding refunds after latest update due to this reason,passage,passage,"game would no longer be supported in unlocked, rooted or jail-broken devices","no longer be supported in unlocked, rooted or jail-broken devices"
971,"Former ""SNL"" star heads to ""House of Lies""",phrase,phrase,Fred Armisen,Fred Armisen
